# Chapter 11 case study: the RAG generator on a Colab TPU

Runs the Chapter 11 TPU measurements for the book. Everything is saved to Google Drive (`MyDrive/HPML_ch11/`), so a disconnect loses nothing.

**Before you start**
1. *Runtime → Change runtime type → **v6e-1 TPU*** (run the notebook again later on **v5e-1 TPU** for the second generation).
2. Add `HF_TOKEN` in the **Secrets** panel (key icon, left sidebar), with notebook access on. Accept the Gemma and Llama 3.1 licences on Hugging Face first.
3. Run the cells in order. Sections 1–4 use PyTorch; Section 5 (JAX) needs a runtime restart first.

**What it measures**
- Gemma-2B-IT on the TPU with TorchTPU: compile time, TTFT for a 512-token RAG prompt, time per output token (TPOT) at batch 1, and logits for the parity check.
- The same at default and at highest matmul precision (Section 11.7 worked example).
- Llama-3.1-8B-Instruct speed on v6e-1 (fits in 32 GB in BF16; skipped on v5e-1).
- Gemma-2B through JAX/KerasHub for the framework comparison.
- Parity of every TPU run against an FP32 CPU reference: KL divergence, top-1 agreement, max relative logit error.

In [ ]:
# 0. Drive, secrets and runtime details
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json, subprocess, platform, shutil
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')        # from the secret store; never typed into a cell
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
RUN = '/content/drive/MyDrive/HPML_ch11'
os.makedirs(f'{RUN}/out', exist_ok=True); os.makedirs(f'{RUN}/ref', exist_ok=True); os.makedirs(f'{RUN}/results', exist_ok=True)
os.chdir(RUN)
tpu_type = os.environ.get('TPU_ACCELERATOR_TYPE') or os.environ.get('COLAB_TPU_ADDR') or 'unknown'
info = {'tpu_type': tpu_type, 'cpu_cores': os.cpu_count(),
        'ram_gb': round(os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1e9, 1),
        'python': platform.python_version(),
        'torch': subprocess.run("python -c 'import torch;print(torch.__version__)'", shell=True, capture_output=True, text=True).stdout.strip()}
print(json.dumps(info, indent=1))
json.dump(info, open(f'results/runtime_{tpu_type}.json', 'w'), indent=1)
TPU = tpu_type.lower()
print('Working folder:', RUN)

## 1. Write the scripts
The same scripts as the book's companion code (`chapter11/`), flattened into one folder.

In [ ]:
%%writefile common.py
"""Shared helpers for the Chapter 11 case study.

Prompts: the 32 RAG prompts of up to 512 tokens from the Chapter 8 workload (JSONL with a "prompt" field).
If that file is not available, build_prompts() makes 32 RAG-style prompts from SQuAD validation contexts so the
scripts can run on their own. The Hugging Face token is read from the environment (HF_TOKEN), which Modal and
Colab provide from their secret stores; it is never written into code or results.
"""
import json, os, platform, subprocess, sys, time

MODEL = os.environ.get("CH11_MODEL", "google/gemma-2b-it")
N_PROMPTS = int(os.environ.get("CH11_N_PROMPTS", "32"))
MAX_PROMPT, NEW_TOKENS, PARITY_POSITIONS = 512, 128, 32


def load_prompts(path=None):
    if path and os.path.exists(path):
        rows = [json.loads(l) for l in open(path)]
        return [r["prompt"] for r in rows][:N_PROMPTS]
    return build_prompts()


def build_prompts():
    from datasets import load_dataset
    ds = load_dataset("rajpurkar/squad", split="validation").select(range(400))
    out, seen = [], set()
    for r in ds:
        if r["context"] in seen:
            continue
        seen.add(r["context"])
        out.append("Answer the question using only the context.\n\nContext:\n" + r["context"] +
                   "\n\nQuestion: " + r["question"] + "\nAnswer:")
        if len(out) == N_PROMPTS:
            break
    return out


def chat(tok, prompt):
    return tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)


def environment(extra=None):
    """Step 1 of the debugging method: record versions that can change results or timings."""
    env = {"python": sys.version.split()[0], "platform": platform.platform(), "time": time.strftime("%Y-%m-%dT%H:%M:%S")}
    for mod in ("torch", "transformers", "vllm", "jax", "jaxlib", "onnxruntime", "onnx", "torch_tpu", "keras", "keras_hub"):
        try:
            m = __import__(mod)
            env[mod] = getattr(m, "__version__", "unknown")
        except Exception:
            pass
    for cmd in (["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], ["rocm-smi", "--showproductname"]):
        try:
            env[cmd[0]] = subprocess.run(cmd, capture_output=True, text=True, timeout=20).stdout.strip()[:300]
        except Exception:
            pass
    env.update({k: v for k, v in os.environ.items() if k.startswith(("TORCH_TPU_", "XLA_", "VLLM_", "JAX_", "CUBLAS_"))})
    env.update(extra or {})
    return env


def merge_json(path, key, value):
    data = json.load(open(path)) if os.path.exists(path) else {}
    data[key] = value
    json.dump(data, open(path, "w"), indent=1)


In [ ]:
%%writefile torchtpu_gemma.py
"""Chapter 11 case study, TPU rows (Code 11.3). Run in a Colab notebook with a TPU runtime (see ch11_tpu_colab.ipynb).

Path A (TorchTPU): PyTorch model on device "tpu", compiled with torch.compile(backend="tpu"), inputs padded to 512.
Path B (JAX): KerasHub Gemma with the JAX backend.
Each path runs twice: default matmul precision and highest precision, for the parity comparison.
Writes results/case_study_tpu.json and out/*.npy; copy them back next to the other results.
HF_TOKEN must be set from Colab's secret store (userdata.get("HF_TOKEN")); never paste it into a cell.
"""
import json, os, sys, time
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import MODEL, MAX_PROMPT, NEW_TOKENS, PARITY_POSITIONS, load_prompts, chat, environment


def torchtpu(ref=None, precision="default"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if precision == "highest":
        torch.set_float32_matmul_precision("highest")   # affects fp32 inputs; BF16 weights are unaffected
    device = torch.device("tpu")
    tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
    dtype = torch.float32 if precision == "highest" else torch.bfloat16
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=dtype).to(device).eval()
    step = torch.compile(model, backend="tpu")
    prompts = [chat(tok, p) for p in load_prompts()]
    sync = torch.accelerator.synchronize

    def fwd(ids):
        with torch.no_grad():
            return step(input_ids=ids).logits

    ids = tok(prompts[:1], return_tensors="pt", padding="max_length", max_length=MAX_PROMPT, truncation=True).input_ids.to(device)
    t = time.perf_counter(); fwd(ids); sync(); cold = time.perf_counter() - t
    t = time.perf_counter(); fwd(ids); sync(); ttft = time.perf_counter() - t
    res = {"compile_s": round(cold - ttft, 1), "ttft_ms": round(ttft * 1e3, 1), "fallback_ops": None,
           "env": environment({"precision": precision})}
    # time per output token: generate() with a static cache (fixed shapes), same method as the GPU rows
    try:
        model.generation_config.cache_implementation = "static"
        def gen(n):
            with torch.no_grad():
                model.generate(input_ids=ids, max_new_tokens=n, min_new_tokens=n, do_sample=False)
            sync()
        gen(NEW_TOKENS)                                   # compile and warm up
        t = time.perf_counter(); gen(1); t1 = time.perf_counter() - t
        t = time.perf_counter(); gen(NEW_TOKENS); tn = time.perf_counter() - t
        res["tpot_b1_ms"] = round((tn - t1) / (NEW_TOKENS - 1) * 1e3, 2)
    except Exception as e:                                # record the failure; it is a finding in its own right
        res["tpot_error"] = repr(e)[:300]
    if ref:
        logits = []
        for pid, toks in zip(ref["prompt_ids"], ref["tokens"]):
            full = torch.tensor([pid + toks[:PARITY_POSITIONS]], device=device)
            lg = fwd(full)[0, len(pid) - 1: len(pid) - 1 + PARITY_POSITIONS].float().cpu().numpy()
            logits.append(lg)
        np.save(f"out/tpu_torchtpu_{os.environ.get('CH11_TAG','gemma2b')}_{precision}.npy", np.stack(logits))
    return res


def jax_keras(ref=None, precision="default"):
    os.environ["KERAS_BACKEND"] = "jax"
    import jax, keras_hub
    jax.config.update("jax_default_matmul_precision", "highest" if precision == "highest" else "default")
    dt = "bfloat16" if precision == "default" else "float32"
    try:
        lm = keras_hub.models.GemmaCausalLM.from_preset("hf://keras/gemma_instruct_2b_en", dtype=dt)
    except Exception:
        lm = keras_hub.models.GemmaCausalLM.from_preset("gemma_instruct_2b_en", dtype=dt)
    # same Gemma chat format as the PyTorch path
    prompts = ["<start_of_turn>user\n" + p + "<end_of_turn>\n<start_of_turn>model\n" for p in load_prompts()]
    lm.compile(sampler="greedy")
    t = time.perf_counter(); lm.generate(prompts[0], max_length=MAX_PROMPT + NEW_TOKENS); cold = time.perf_counter() - t
    t = time.perf_counter(); lm.generate(prompts[0], max_length=MAX_PROMPT + NEW_TOKENS); warm = time.perf_counter() - t
    lm.generate(prompts[0], max_length=MAX_PROMPT + 1)
    t = time.perf_counter(); lm.generate(prompts[0], max_length=MAX_PROMPT + 1); t1 = time.perf_counter() - t
    res = {"compile_s": round(cold - warm, 1), "ttft_ms": round(t1 * 1e3, 1),
           "tpot_b1_ms": round((warm - t1) / (NEW_TOKENS - 1) * 1e3, 2), "env": environment({"precision": precision})}
    if ref:
        logits = []
        for pid, toks in zip(ref["prompt_ids"], ref["tokens"]):
            x = np.array([pid + toks[:PARITY_POSITIONS]], dtype=np.int32)
            out = lm.backbone({"token_ids": x, "padding_mask": np.ones_like(x)})
            lg = lm.backbone.token_embedding(out, reverse=True)
            logits.append(np.asarray(lg[0, len(pid) - 1: len(pid) - 1 + PARITY_POSITIONS], dtype=np.float32))
        np.save(f"out/tpu_jax_{os.environ.get('CH11_TAG','gemma2b')}_{precision}.npy", np.stack(logits))
    return res


if __name__ == "__main__":
    os.makedirs("out", exist_ok=True)
    ref = None
    if os.path.exists("ref/ref_ids.json"):
        ref = json.load(open("ref/ref_ids.json")); ref["tokens"] = np.load("ref/ref_tokens.npy").tolist()
    which = sys.argv[1] if len(sys.argv) > 1 else "torchtpu"
    precs = sys.argv[2].split(",") if len(sys.argv) > 2 else ["default", "highest"]
    tag = os.environ.get("CH11_TAG", "gemma2b")
    out = {}
    for prec in precs:
        try:
            out[f"tpu_{which}_{tag}_{prec}"] = (torchtpu if which == "torchtpu" else jax_keras)(ref, prec)
        except Exception as e:   # a failure on a new platform is a result: record it and continue
            import traceback
            out[f"tpu_{which}_{tag}_{prec}"] = {"error": repr(e)[:500], "trace": traceback.format_exc()[-2000:], "env": environment()}
    json.dump(out, open(f"case_study_tpu_{which}_{tag}.json", "w"), indent=1)
    print(json.dumps(out, indent=1))


In [ ]:
%%writefile cpu_reference.py
"""FP32 CPU reference for parity: greedy tokens and teacher-forced logits at the first 32 generated positions.

Writes ref_cpu_fp32.npy (logits), ref_tokens.npy (128 greedy tokens per prompt) and ref_ids.json (prompt ids),
which every other platform reuses so that all logits are computed on the same token sequences.
Runs on a CPU in about an hour for 32 prompts; run it once on Modal (cpu=16) or locally.
"""
import json, os, sys
import numpy as np, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import MODEL, MAX_PROMPT, NEW_TOKENS, PARITY_POSITIONS, load_prompts, chat, environment
from transformers import AutoModelForCausalLM, AutoTokenizer


def main(prompts_path=None, out="ref"):
    os.makedirs(out, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).eval()
    ids_all, toks, logits = [], [], []
    for p in load_prompts(prompts_path):
        ids = tok(chat(tok, p), return_tensors="pt", truncation=True, max_length=MAX_PROMPT).input_ids
        with torch.no_grad():
            g = model.generate(ids, max_new_tokens=NEW_TOKENS, do_sample=False, min_new_tokens=NEW_TOKENS)
            full = g[:, : ids.shape[1] + PARITY_POSITIONS]
            lg = model(full).logits[0, ids.shape[1] - 1: ids.shape[1] - 1 + PARITY_POSITIONS]
        ids_all.append(ids[0].tolist()); toks.append(g[0, ids.shape[1]:].numpy()); logits.append(lg.float().numpy())
    np.save(f"{out}/ref_cpu_fp32.npy", np.stack(logits)); np.save(f"{out}/ref_tokens.npy", np.stack(toks))
    json.dump({"prompt_ids": ids_all, "env": environment()}, open(f"{out}/ref_ids.json", "w"))


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else None)


In [ ]:
%%writefile compare_logits.py
"""Model-level parity check (Code 11.5): compare saved logits from a candidate platform with the FP32 reference.

Each .npy file holds float32 logits of shape [prompts, positions, vocab] for the same prompts and the same
reference token sequence (teacher-forced on the reference's greedy tokens, so positions line up).
Usage: python compare_logits.py ref_cpu_fp32.npy cand.npy [--greedy ref_tokens.npy cand_tokens.npy] --key NAME
"""
import argparse, json, os, sys
import numpy as np


def log_softmax(x):
    x = x - x.max(-1, keepdims=True)
    return x - np.log(np.exp(x).sum(-1, keepdims=True))


def compare(ref, cand):
    lp, lq = log_softmax(ref.astype(np.float64)), log_softmax(cand.astype(np.float64))
    kl = (np.exp(lp) * (lp - lq)).sum(-1)
    return {"kl_mean": float(kl.mean()), "kl_p99": float(np.quantile(kl, 0.99)),
            "top1_agreement": float((ref.argmax(-1) == cand.argmax(-1)).mean()),
            "max_rel_logit_err": float(np.abs(cand - ref).max() / np.abs(ref).max())}


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("ref"); ap.add_argument("cand"); ap.add_argument("--key", required=True)
    ap.add_argument("--greedy", nargs=2, help="token id arrays [prompts, 128] from free-running greedy generation")
    ap.add_argument("--out", default="results/parity.json")
    a = ap.parse_args()
    r = compare(np.load(a.ref), np.load(a.cand))
    if a.greedy:
        g0, g1 = np.load(a.greedy[0]), np.load(a.greedy[1])
        # identical prefix length per prompt, averaged
        same = (g0 == g1)
        prefix = [int(np.argmin(s)) if not s.all() else s.size for s in same]
        r["greedy_identical_tokens"] = float(np.mean(prefix))
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from common import merge_json
    merge_json(a.out, a.key, r)
    print(json.dumps(r, indent=1))


## 2. FP32 CPU reference (once; reused by every platform)
Runs Gemma-2B in FP32 on the TPU host's CPU: greedy tokens for 32 RAG prompts and the logits at the first 32 generated positions. It is skipped if the files already exist on Drive. Expect 20–60 minutes depending on the host's CPU count.

The prompts are RAG prompts built from SQuAD validation contexts (question plus retrieved passage). To use the exact Chapter 8 prompts, upload `ch08_rag_prompts.jsonl` (one `{"prompt": ...}` per line) to the Drive folder and pass its name as an argument.

In [ ]:
import os
if os.path.exists('ref/ref_cpu_fp32.npy'):
    print('Reference already on Drive; skipping.')
else:
    !pip -q install -U transformers datasets accelerate
    !python cpu_reference.py $( [ -f ch08_rag_prompts.jsonl ] && echo ch08_rag_prompts.jsonl ) 2>&1 | tail -5
!ls -la ref

## 3. TorchTPU: Gemma-2B (default and highest precision)
The first call prints the cold compile time. If a cell fails, the error is saved in the results JSON, because a failure on a new platform is a result for the chapter. For op-level errors, rerun with `TORCH_TPU_LAUNCH_BLOCKING=1` (debug mode) and record what it shows.

In [ ]:
!pip -q install torch_tpu 2>&1 | tail -2
!python -c "import torch, torch_tpu; print(torch.__version__, torch_tpu.__version__)" || echo 'torch_tpu not importable: record this and continue with Section 5 (JAX)'
%env CH11_TAG=gemma2b
%env CH11_MODEL=google/gemma-2b-it
!python torchtpu_gemma.py torchtpu default,highest 2>&1 | tail -40

## 4. TorchTPU: Llama-3.1-8B-Instruct, speed only (v6e-1 only)
Llama-3.1-8B in BF16 takes about 16 GB, so it fits in the v6e's 32 GB and not in the v5e's 16 GB. This run records compile time, TTFT and TPOT at default precision; parity for this model is not computed (its FP32 CPU reference would take several hours).

In [ ]:
if 'v6e' in TPU or 'v6' in TPU:
    os.environ['CH11_TAG'] = 'llama8b'; os.environ['CH11_MODEL'] = 'meta-llama/Llama-3.1-8B-Instruct'
    if os.path.isdir('ref'): os.rename('ref', 'ref_gemma')      # no reference for Llama: skip parity
    !python torchtpu_gemma.py torchtpu default 2>&1 | tail -30
    if os.path.isdir('ref_gemma'): os.rename('ref_gemma', 'ref')
else:
    print('Not a v6e runtime (TPU type:', TPU, '): skipping Llama-3.1-8B.')

## 4b. Parity of the TorchTPU runs against the FP32 reference

In [ ]:
import glob
for f in sorted(glob.glob('out/tpu_torchtpu_*.npy')):
    key = os.path.basename(f)[:-4] + '_' + TPU.replace('/', '-')
    !python compare_logits.py ref/ref_cpu_fp32.npy {f} --key {key} | tail -8

## 5. JAX / KerasHub: Gemma-2B
**Restart the runtime first** (*Runtime → Restart session*), so only JAX holds the TPU. Then run Section 0 again and continue here.

In [ ]:
!pip -q install -U keras keras-hub "jax[tpu]" 2>&1 | tail -2
%env CH11_TAG=gemma2b
%env KERAS_BACKEND=jax
!python torchtpu_gemma.py jax default,highest 2>&1 | tail -40
import glob
for f in sorted(glob.glob('out/tpu_jax_*.npy')):
    key = os.path.basename(f)[:-4] + '_' + TPU.replace('/', '-')
    !python compare_logits.py ref/ref_cpu_fp32.npy {f} --key {key} | tail -8

## 6. Collect results
Writes one summary file and a zip of everything (no model weights, no token). Download the zip from Drive (`MyDrive/HPML_ch11/`) and send it back.

In [ ]:
import glob, json, zipfile, time
summary = {'runtime': info}
for f in glob.glob('case_study_tpu_*.json'): summary[os.path.basename(f)] = json.load(open(f))
if os.path.exists('results/parity.json'): summary['parity'] = json.load(open('results/parity.json'))
name = f"results/ch11_tpu_{TPU.replace('/', '-')}_{time.strftime('%Y%m%d_%H%M')}.json"
json.dump(summary, open(name, 'w'), indent=1)
# guard: no token in anything we ship
for f in glob.glob('**/*.json', recursive=True):
    assert 'hf_' not in open(f).read(), f'possible token in {f}'
z = name.replace('.json', '.zip')
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in [name] + glob.glob('case_study_tpu_*.json') + glob.glob('results/*.json'):
        zf.write(f)
print('Saved', z)
print(json.dumps({k: v for k, v in summary.items() if k != 'parity'}, indent=1)[:3000])